# Contrastive training on Colab T4

Trains the dual-encoder model in `backend/services/contrastive.py` on the LLM-labeled triples and produces the inference artifacts the backend serves.

## What you need to upload
When prompted by the upload cell, drag in two files from your local `backend/cache/`:
1. `contrastive_triples.jsonl` (~3 MB)
2. `profiles_2023-01-01_2024-12-31.npz` (~90 KB)

## What gets downloaded
At the end the notebook downloads:
- `text_proj_state.pt`, `city_encoder_state.pt` — trained weights
- `city_scaler.npz` — StandardScaler params (mean, scale)
- `city_embeddings_learned.npz` — final 230x32 city embedding cache
- `contrastive_test_split.jsonl` — held-out split for reproducibility

Drop those into your local `backend/cache/`. Runtime: enable T4 GPU (Runtime -> Change runtime type -> T4 GPU). End-to-end ~2 min.

If GPU is unavailable, this still works on CPU (precompute ~2 min, training ~3 min).

In [ ]:
# 1. Clone the repo
!rm -rf weather-ml-v2
!git clone --depth=1 https://github.com/YardenMorad2003/weather-ml-v2.git
%cd weather-ml-v2

In [ ]:
# 2. Install sentence-transformers (Colab already ships torch + numpy)
!pip install -q sentence-transformers

In [ ]:
# 3. Upload the two cache files from your local backend/cache/
import os
os.makedirs('backend/cache', exist_ok=True)

from google.colab import files
uploaded = files.upload()  # contrastive_triples.jsonl AND profiles_2023-01-01_2024-12-31.npz
for fname in uploaded:
    os.replace(fname, f'backend/cache/{fname}')
print('cache contents:', os.listdir('backend/cache'))

In [ ]:
# 4. Precompute MiniLM embeddings over the 10k labeled queries (one-time).
!python -u -m backend.scripts.precompute_embeddings \
    --in backend/cache/contrastive_triples.jsonl \
    --out backend/cache/text_embeddings.npz

In [ ]:
# 5. Train. ~30s on T4, a few min on CPU.
!python -u -m backend.scripts.train_contrastive \
    --triples backend/cache/contrastive_triples.jsonl \
    --text-emb backend/cache/text_embeddings.npz \
    --profiles backend/cache/profiles_2023-01-01_2024-12-31.npz \
    --out-dir backend/cache \
    --epochs 100 --batch-size 256 --lr 1e-3 --seed 42 --eval-every 10

In [ ]:
# 6. Bundle the trained artifacts and download. Drop the contents into
# your local backend/cache/ to make the model loadable from the backend.
import os, zipfile
outs = [
    'text_proj_state.pt',
    'city_encoder_state.pt',
    'city_scaler.npz',
    'city_embeddings_learned.npz',
    'contrastive_test_split.jsonl',
]
with zipfile.ZipFile('contrastive_artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in outs:
        path = f'backend/cache/{fn}'
        if os.path.exists(path):
            z.write(path, arcname=fn)
            print('  bundled', fn)
        else:
            print('  MISSING', path)

from google.colab import files
files.download('contrastive_artifacts.zip')
